# E17 MagNet Composite Graphs

Author: Arush Arora

## Introduction

The current codebase focuses on delivering text $\mathbf{X}$ and positional encodings $\Psi$ through separate channels to the LLM, which seems to be confounding their interleaving process. $\Psi$, the GREPs, are defined as follows:

$$\Phi = \Phi\big(\mathbf{q};\, \mathbf{S}, \mathcal{H}\big) \qquad \mathbf{P} \coloneqq \mathbb{E}_{\mathbf{q} \sim \mathcal{N}(0,\, \mathbf{I}_D)}\big[\Phi\big] \qquad \mathbf{C} \coloneqq \mathbb{E}_{\mathbf{q}}\big[\Phi\Phi^\top\big] - \mathbf{P}\mathbf{P}^\top$$

$$\mathbf{\Psi} = \Phi\big(\mathbf{X} + \mathbf{P};\, \mathcal{T}\big) \quad \text{or} \quad \mathbf{\Psi} = \Phi\bigg((\mathbf{I}_{n + c} + \mathbf{C})\begin{bmatrix}\mathbf{X} \\ \mathbf{P}\end{bmatrix};\, \mathcal{T}\bigg)$$

In this experiment, we wish to work with the Composite Graph paradigm to determine whether the MagNet architecture from the paper [“MagNet: A Neural Network for Directed Graphs” (Zhang et al., 2021)](https://arxiv.org/pdf/2102.11391) can encode the directional spectral information essential to the full Composite Graphs architecture to correct the issues that were appearing during the latest iteration of the experiment in June.

Specifically, we propose the following instantiation of $\mathbf{S}$ per the paper, the normalized complex Hermitian adjacency matrix $\bar{\mathbf{H}}^{(r)}$, which can also be seen as the Normalized Magnetic Adjacency:

$$\mathbf{S} = \bar{\mathbf{H}}^{(r)} \coloneqq \mathbf{D}^{-1/2} \mathbf{A} \mathbf{D}^{-1/2} \odot \exp\big(i\mathbf{\Theta}^{(r)}\big)$$

$$\mathbf{\Theta}^{(r)} \coloneqq 2 \pi r (\mathbf{A} - \mathbf{A}^\top), \quad r \ge 0$$

We thus define the Composite Graphs architecture as such:

$$\big|\mathcal{V}_\text{Tx}\big| = c, \qquad \big|\mathcal{V}_\text{Sc}\big| = n$$

$$\mathcal{G} = \big(\mathcal{V}_\text{Tx} \cup \mathcal{V}_\text{Sc}, \mathcal{E}_\text{Tx} \cup \mathcal{E}_\text{Tx} \cup \mathcal{V}_\text{Cross} \cup \mathcal{V}_\text{Mention})$$

$$\mathcal{E}_\text{Cross} = \big\{\{u, v\} : \text{$u$ is in the node label for $v$},\ u \in \mathcal{V}_\text{Tx},\ v \in \mathcal{V}_\text{Sc}\big\}$$

$$\mathcal{E}_\text{Mention} = \big\{\{u, v\} : \text{$u$ and $v$ are in same node labels},\ u, v \in \mathcal{V}_\text{Tx}\big\}$$

## Mathematical Overview

### The R-PEARL GNN

The Random Positional Encoding (R-PEARL) GNN architecture is a PE generator that inputs white noise and processes it over an undirected graph $\mathcal{G} = (\mathcal{V}, \mathcal{E}, \mathcal{W})$. In this work, the graph is represented by an adjacency matrix $A$, and the GNN composes [Topology Adaptive Graph (TAG)](https://arxiv.org/abs/1710.10370) Convolutional Layers with pointwise nonlinearities (demodulators).

#### Graph Convolutional Network (GNN)

The code below establishes this project's implementation of a Graph Convolutional Network, which is the foundational architecture comprising R-PEARL. The equation to demonstrate the internal architecture of this NN as follows (in most cases, $\mathbf{P}(\cdot) = \mathbf{I}(\cdot)$, where $\mathbf{I}$ is the identity function):
$$\Phi(\mathbf{X}, \mathbf{S}, \mathcal{H}) = \mathbf{X}^{(L)}$$
$$\mathbf{X}^{(0)} = \mathbf{X} \qquad \mathbf{X}^{(l)} = \mathbf{P}\Bigg[\sigma\Bigg(\sum_{k = 0}^{K^{(l)} - 1} \mathbf{S}^k\mathbf{X}^{(l - 1)}\mathbf{H}_k^{(l)}\Bigg)\Bigg]$$

#### Random Graph Positional Encodings (R-PEARL)

The R-PEARL architecture extends on the GCN by instantiating it with simply one layer – a TAG Convolution and Demodulator. The mathematical equations below express the functionality of the R-PEARL network:
1. The white-noise matrix is sampled from the Gaussian distribution. $$\mathbf{Q} \in \mathbb{R}^{M \times N} \qquad \mathbf{Q} \sim \mathcal{N}(0, \mathbf{I}) \qquad \mathbf{Q} = \begin{bmatrix}
  \mathbf{q}^{(0)} & \cdots & \mathbf{q}^{(m)} & \cdots & \mathbf{q}^{(M)}
  \end{bmatrix}$$

2. The R-PEARL network has row-vector parameter $\mathbf{H}^{(0)} \in \mathbb{R}^{1 \times D}$. It takes in each column of the white-noise matrix individually and produces a sample $\mathbf{P}^{(m)} \in \mathbb{R}^{N \times D}$, which are then pooled to form GREP $\mathbf{P}$:$$\mathbf{P}^{(m)} = \Phi\Big(\mathbf{q}^{(m)};\, \mathbf{S}, \mathcal{H}\Big) = \sigma\bigg(\sum_{k = 0}^{K = 1} \mathbf{S}^k\mathbf{q}^{(m)} {\mathbf{H}}_k\bigg)$$ $$\mathbf{P} = \mathbb{\hat{E}}\Big[\mathbf{p}^{(m)}\Big] = \frac{1}{M}\sum_{m = 1}^{M} \mathbf{P}^{(m)}$$

### Sparse Graph Transformer

The Sparse Graph Transformer (hereafter named Graph Transformer or GT) follows the same architecture as that of a normal transformer, albeit that the attention mecahnism is modified to scope only over the $k$-hop neighborhood of the query node. The mathematical equations below express the functionality of the Graph Transformer:
$$\mathbf{X}_L = \Phi\Big(\mathbf{X}_0 + \mathbb{\hat{E}}_{\mathbf{q \sim \mathcal{N}(0,\, \mathbf{I})}}\big[\Phi(\mathbf{q};\, S, \mathcal{H})\big];\, \mathcal{T}\Big)$$
$$\mathbf{A}^{(h)}_l = \left[\frac{\exp\left[\left(\mathbf{Q}^{(h)}_l \mathbf{x}_{l-1,\ t}\right)^\top \left(\mathbf{K}^{(h)}_l \mathbf{X}_{l-1,\ U}\right)\right]}{\mathbf{1}^\top \exp\left[\left(\mathbf{Q}^{(h)}_l \mathbf{x}_{l-1,\ t}\right)^\top \left(\mathbf{K}^{(h)}_l \mathbf{X}_{l-1,\ U}\right)\right]}\right]^\top_{\begin{subarray}{l}t \in [N] \\[2.5pt] U = \mathcal{N}^{\le k}(t)\end{subarray}}$$
$$\mathbf{Y}^{(h)}_l = \left(\mathbf{W}_o\right)^\top_l \mathbf{V}_l \mathbf{X}_{l - 1} \left(\mathbf{A}^{(h)}_l\right)^\top$$
$$\mathbf{X}_l = \sigma\bigg(\sum_{h = 1}^H \mathbf{Y}^{(h)}_l\bigg)$$

### Transformer

The Transformer architecture follows that of the Llama3.1-8B distilled PRISM model. First, the TXT file, containing the scene-graph data, is tokenized and embedded into matrices $E$ and $\tilde{X}$ as follows, where $V$ is the size of the vocabulary and $d$ is the embedding dimension.

$$\text{TXT Tokenized Data from GPT-4: } E = \begin{bmatrix}
\mathbf{e}_1 & \mathbf{e}_2 & \overset{\mathbf{e}_t}{\cdots} & \mathbf{e}_T
\end{bmatrix}^\top \qquad \mathbf{e}_t \in \mathbb{R}^V$$

$$\text{Embed: } X = \begin{bmatrix}
\mathbf{x}_1 & \mathbf{x}_2 & \overset{\mathbf{x}_t}{\cdots} & \mathbf{x}_T
\end{bmatrix}^\top \qquad \mathbf{x}_t \in \mathbb{R}^d$$

Next, the transformer operates using the equations below:

$$\mathbf{X} = \mathbf{\tilde{X}} + \mathbf{P}$$

$$\mathbf{Z}_{1:t}^{(L)} = \operatorname{Trf}\Big(\mathbf{X}_{1:t}, {\mathcal{T}}_l\Big) \qquad {\mathcal{T}}_l = \begin{bmatrix}
\mathbf{Q}_l & \mathbf{K}_l & \mathbf{V}_l & \left(\mathbf{W}_o\right)_l
\end{bmatrix}^\top \in \mathbb{R}^{4 \times T \times D}$$

$$\hat{\mathbf{Y}}_{t + 1} = \operatorname{Linear}\Big(\mathbf{Z}_{1:t}^{(L)}\Big) \in \mathbb{R}^V$$
$$\text{Cross-Entropy Loss: } \mathcal{L}(E, \hat{\mathbf{Y}}) = \sum_t \sum_v e_{vt}\log{\hat{y}_t}$$

### Graph-Augmented LLM

The last class that is needed to create the full GREP-PRISM architecture is the `GraphAugmentedLLM`, which simply implements the following equation as a Neural Network object in PyTorch's `torch.nn` module (referring to above equations for definitions).
$$\mathbf{P} = \mathbb{\hat{E}}\Big[\mathbf{p}^{(m)}\Big] = \frac{1}{M}\sum_{m = 1}^{M} \Phi\Big(\mathbf{q}^{(m)}, \mathbf{S}, \mathcal{H}\Big)$$

$$\mathbf{X} = \mathbf{\tilde{X}} + \mathbf{P}$$

$${\mathbf{Z}}_{1:t}^{(L)} = \operatorname{Trf}\Big({\mathbf{X}}_{1:t}; \, \cdot \,\Big)$$

## Setup

In [1]:
# %env CUDA_VISIBLE_DEVICES=0
%load_ext autoreload
%autoreload 2

In [2]:
# Import modules.
import gc
import math
import copy
import wandb
import torch
import random
import pickle
import sympy as sp
import numpy as np
import networkx as nx

from typing import Union, Optional

import torch
from torch import Tensor, nn
import matplotlib.pyplot as plt
from IPython.display import display
from torch_geometric.data import Data
from torch_geometric.nn import ChebConv
from torch.nn.utils import clip_grad_norm_
from torch_geometric.typing import OptTensor
from torch_geometric.utils import to_networkx
from torch_geometric.loader import DataLoader
from torch.distributions import Cauchy, Normal
from torch_geometric.nn.dense.linear import Linear
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.utils import to_dense_adj, to_networkx
from torch_geometric.utils.num_nodes import maybe_num_nodes
from torch_geometric.utils import coalesce, remove_self_loops

from prism.models.gt import GraphTransformer, SemanticGraphTransformer
from prism.models.r_pearl import RandomGNNPositionalEncodings
from prism.data import data, utils

In [3]:
# Weights & Biases setup. Mirrors prism.training.train_v3._setup_wandb (project /
# name / tags / group + full-config logging), adapted for this notebook's hand-written
# train loops. Each training stage gets its own run, grouped/tagged by GNN type so the
# R-PEARL and GT variants of the same stage line up on one W&B dashboard. The helpers
# introspect the live optimizer / scheduler / loss objects so EVERY hyperparameter is
# logged without hand-maintaining a list.
WANDB_PROJECT = 'e17-composite-graphs'


def optimizer_hparams(optimizer):
    """Every optimizer setting: class name, shared defaults, and per-param-group values
    (LRs, betas, eps, weight_decay, ...) with the parameter tensors stripped out."""
    return {
        'optimizer': type(optimizer).__name__,
        'optimizer_defaults': dict(optimizer.defaults),
        'param_groups': [
            {k: v for k, v in g.items() if k != 'params'}
            for g in optimizer.param_groups
        ],
    }


def scheduler_hparams(scheduler):
    """Every LR-scheduler setting (or {'scheduler': None} when unused)."""
    if scheduler is None:
        return {'scheduler': None}
    keys = ('mode', 'factor', 'patience', 'threshold', 'threshold_mode',
            'cooldown', 'min_lrs', 'eps')
    return {
        'scheduler': type(scheduler).__name__,
        **{k: getattr(scheduler, k) for k in keys if hasattr(scheduler, k)},
    }


def loss_hparams(loss_fn):
    """Loss class, reduction, and pos_weight (resolved to plain Python)."""
    out = {'loss_fn': type(loss_fn).__name__,
           'reduction': getattr(loss_fn, 'reduction', None)}
    pos_weight = getattr(loss_fn, 'pos_weight', None)
    if pos_weight is not None:
        out['pos_weight'] = (pos_weight.detach().cpu().tolist()
                             if torch.is_tensor(pos_weight) else pos_weight)
    return out


def init_wandb(stage, hparams):
    """Start a W&B run for a training `stage`.

    Logs the FULL run config: the GNN construction kwargs (`model_hparams`, set in the
    GNN-instantiation cell) plus every optimizer / scheduler / loss / batching
    hyperparameter the caller assembles in `hparams`. `model_type` selects R-PEARL vs
    GT and drives the run name / tag / group. Returns the run; `reinit=True` so
    successive stages in one notebook session each open a fresh run.
    """
    return wandb.init(
        project=WANDB_PROJECT,
        name=f'{stage}_{model_type}',
        tags=[stage, model_type],
        group=model_type,
        config={'model_type': model_type, 'stage': stage,
                'model': model_hparams, **hparams},
        reinit='return_previous',
    )

In [4]:
# Define a tensor rendering function.
def render_matrix(mat: torch.tensor, sig_figs: int = 3, decimals: int = 0):
    out = sp.Matrix(mat.detach().cpu().numpy())
    if sig_figs > 0:
        return sp.N(out, sig_figs)
    if decimals > 0:
        return out.applyfunc(lambda x: x.round(decimals))
    return out

In [5]:
# Standard options.
gnn_path  = '/home/shared/GREP-PRISM/outputs/e13f_alpha_khops/e13f_alpha00_ai8c2bm0/gnn_weights.pt'
ex_path   = '../data/n_30/gen/nav100_n30_gemma_data/split/test_graphs'
eval_path = '../data/n_100/gen/nav_n100_gemma_data/test_graphs'
save_path = '../data/pickle/e6_eval_graphs.pkl'
device    = 'cuda'

In [6]:
# Setup eval infrastructure.
samples_by_graph, graph_file_by_name = data.load_samples_by_graph(ex_path)
graph_file = random.choice(list(samples_by_graph.keys()))
eval_data = samples_by_graph[graph_file]
eval_data = {graph_file: [random.choice(eval_data)]}

In [7]:
print(f'/Users/cyberlives/Documents/GitHub/GREP-PRISM/eval/render/revised/{graph_file}.html')

/Users/cyberlives/Documents/GitHub/GREP-PRISM/eval/render/revised/data_gen_016.html


## Experiments

### §1 Pretraining a GNN to Classify Text-Scene Node Pairings

We first hope to optimize a GNN (R-PEARL or Graph Transformer) to classify crosslink edges of the form $\{u, v\}$ where $u \in \mathcal{V}_\text{Tx}$ and $v \in \mathcal{V}_\text{Sc}$. Such a model will serve as a backbone pretrained model for fine-tuning on identifying node families given a node (including all other bucket text nodes, mention text nodes, and scene nodes). This architecture will instantiate the already implemented `GCN`, `RandomGNNPositionalEncodings`, and `GraphTransformer` architectures with the Magnetic Adjacency $\bar{\mathbf{H}}^{(r)}$ defined above.

#### Model Definitions

We first construct the MagNet architecture, mathematized below:

##### CReLU and Complex Linear Transformation

$$z \in \mathbb{C} \qquad \mathbf{X} \in \mathbb{C}^{N \times D}$$

$$\sigma(z) = \begin{cases}
    z, & \text{if } \arg(z) \in [- \pi / 2, \pi / 2] \\
    0, & \text{otherwise}
\end{cases}$$

$$\mathrm{CLin}_\phi\big(\mathbf{X}\big) = \mathrm{Lin}_\phi\Big(\mathrm{Re}\big(\mathbf{X}\big)\Big) + i\bigg[\mathrm{Lin}_\phi\Big(\mathrm{Im}\big(\mathbf{X}\big)\Big)\bigg]$$

In [8]:
def crelu(x: Tensor) -> Tensor:
    """Applies the complex rectified linear unit function."""
    return x * (x.real >= 0)


def clin(lin: Linear, x: Tensor) -> Tensor:
    """Applies a real-valued linear transformation to a complex tensor."""
    return torch.complex(lin(x.real), lin(x.imag))

##### Magnetic Laplacian and Chebyshev Convolution

$$A_s = \frac{1}{2}(\mathbf{A} + \mathbf{A}^\top)$$

$$\mathbf{\bar{H}}^{(r)} \coloneqq \mathbf{D}_s^{-1/2} \mathbf{A}_s \mathbf{D}_s^{-1/2} \odot \exp \left( i \mathbf{\Theta}^{(r)} \right)$$

$$\mathbf{\Theta}^{(r)} = 2 \pi r \, \mathrm{sgn} (\mathbf{A} - \mathbf{A}^\top), \quad r \ge 0$$

$$\bar{\mathbf{L}}^{(r)} \coloneqq \mathbf{I}_N - \bar{\mathbf{H}}^{(r)}$$

---

$$\mathbf{X}^{(0)} \in \mathbb{R}^{N \times D} \qquad \mathbf{S} = \mathbf{\bar{L}}^{(r)}$$

$$\mathbf{Y}^{(l)} = \sum_{k=1}^{K} \mathbf{Z}_k(\mathbf{X}^{(l)};\, \mathbf{S}) \mathbf{H}_k$$

$$\mathbf{S} \in \mathbb{C}^{N \times N} \quad \mathrm{spec}(\mathbf{S}) \subseteq [-1, 1]$$

$$\begin{align*}
\mathbf{Z}_1 &= \mathbf{X} \\
\mathbf{Z}_2 &= \mathbf{S} \mathbf{X} \\
\mathbf{Z}_k &= 2 \cdot \mathbf{S} \mathbf{Z}_{k-1} - \mathbf{Z}_{k-2}
\end{align*}$$

$$\mathbf{H}_k \in \mathbb{R}^{F \times G}$$

In [9]:
class MagChebConv(ChebConv):
    """The magnetic Chebyshev spectral graph convolutional operator."""
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        K: int,
        normalization: Optional[str] = 'sym',
        bias: bool = True,
        r: float = 0.25,
        learn_r: bool = False,
        phase: str = 'binary',
        **kwargs,
    ):
        super().__init__(in_channels, out_channels, K, normalization, bias,
                         **kwargs)

        assert phase in ['binary', 'weight'], 'Invalid phase'

        self.phase, self.r_const = phase, r
        self.r_logit = nn.Parameter(
            torch.tensor(min(max(r, 1e-3), 0.249) / 0.25).logit(),
        ) if learn_r else None

    @property
    def r(self) -> Tensor:
        r"""The charge parameter :math:`r`, constrained to :math:`[0, 0.25]`.
        A hard clamp would leave no gradient at the upper end.
        """
        return self.r_const if self.r_logit is None else 0.25 * self.r_logit.sigmoid()

    def __norm__(
        self,
        edge_index: Tensor,
        num_nodes: Optional[int],
        edge_weight: OptTensor,
        normalization: Optional[str],
        lambda_max: OptTensor = None,
        dtype: Optional[int] = None,
        batch: OptTensor = None,
    ):
        num_nodes = maybe_num_nodes(edge_index, num_nodes)
        edge_index, edge_weight = remove_self_loops(edge_index, edge_weight)
        if edge_weight is None:
            edge_weight = torch.ones(edge_index.size(1), dtype=dtype,
                                     device=edge_index.device)

        row, col = edge_index[0], edge_index[1]
        sgn = edge_weight if self.phase == 'weight' else torch.ones_like(edge_weight)
        edge_index = torch.stack([torch.cat([row, col]), torch.cat([col, row])])
        edge_attr = torch.stack([edge_weight.repeat(2), torch.cat([sgn, -sgn])], 1)
        edge_index, edge_attr = coalesce(edge_index, edge_attr, num_nodes)

        edge_index, edge_weight = gcn_norm(edge_index, edge_attr[:, 0] / 2,
                                           num_nodes, add_self_loops=False)
        theta = (2 * math.pi) * self.r * edge_attr[:, 1]

        return edge_index, torch.complex(edge_weight * theta.cos(),
                                         edge_weight * theta.sin())

    def forward(
        self,
        x: Tensor,
        edge_index: Tensor,
        edge_weight: OptTensor = None,
        batch: OptTensor = None,
        lambda_max: OptTensor = None,
    ) -> Tensor:

        edge_index, norm = self.__norm__(
            edge_index,
            x.size(self.node_dim),
            edge_weight,
            self.normalization,
            lambda_max,
            dtype=x.real.dtype,
            batch=batch,
        )

        Tx_0 = x
        Tx_1 = x  # Dummy.
        out = clin(self.lins[0], Tx_0)

        # propagate_type: (x: Tensor, norm: Tensor)
        if len(self.lins) > 1:
            Tx_1 = self.propagate(edge_index, x=x, norm=norm)
            out = out + clin(self.lins[1], Tx_1)

        for lin in self.lins[2:]:
            Tx_2 = self.propagate(edge_index, x=Tx_1, norm=norm)
            Tx_2 = 2. * Tx_2 - Tx_0
            out = out + clin(lin, Tx_2)
            Tx_0, Tx_1 = Tx_1, Tx_2

        if self.bias is not None:
            out = out + self.bias

        return out

##### MagNet

$$\Phi\Big(\mathbf{X};\, \mathbf{\bar{L}}^{(r)}, \mathcal{H}\Big) = \mathbf{X}'$$

$$\mathbf{X}^{\prime} = \mathbf{W}^\top \left[ \mathrm{Re}\Big(\mathbf{X}^{(L)}\Big) \, \Vert \,
        \mathrm{Im}\Big(\mathbf{X}^{(L)}\Big) \right]$$

$$\mathbf{X}^{(\ell)} = \sigma \Big( \mathbf{Y}^{(\ell - 1)} \Big)$$

In [10]:
class MagNet(nn.Module):
    """The magnetic graph neural network."""
    def __init__(
        self,
        in_channels: int,
        hidden_channels: int,
        num_layers: int,
        skip_connection: bool = False,
        dropout: float = 0.5,
        k: int = 3,
        r: float = 0.25,
        learn_r: bool = False,
        phase: str = 'binary',
    ):
        super().__init__()

        assert num_layers >= 2, 'MagNet requires at least 2 layers'

        dims = [in_channels] + [hidden_channels] * (num_layers - 1)
        self.convs = nn.ModuleList([
            MagChebConv(i, hidden_channels, k, r=r, learn_r=learn_r,
                        phase=phase) for i in dims
        ])
        self.norms = nn.ModuleList(
            [nn.LayerNorm(hidden_channels) for _ in range(num_layers - 2)])
        self.unwind = nn.Linear(2 * hidden_channels, hidden_channels)
        self.dropout = nn.Dropout(dropout)
        self.skip_connection = skip_connection
        self.embedding_dim = hidden_channels

    @property
    def r(self) -> Tensor:
        r"""The charge parameter :math:`r` of each layer."""
        return torch.stack([torch.as_tensor(conv.r) for conv in self.convs])

    def forward(self, data: Data) -> Tensor:
        device = next(self.parameters()).device
        x, edge_index = data.x.to(device), data.edge_index.to(device)
        edge_weight = getattr(data, 'edge_weight', None)
        if edge_weight is not None:
            edge_weight = edge_weight.to(device)
        batch = getattr(data, 'batch', None)
        if batch is not None:
            batch = batch.to(device)

        x = x_prev = x + 0j
        for i, conv in enumerate(self.convs[:-1]):
            x = conv(x_prev, edge_index, edge_weight, batch)
            if i < len(self.norms):
                x = torch.complex(self.norms[i](x.real), self.norms[i](x.imag))
            x = crelu(x) * self.dropout(torch.ones_like(x.real))
            if self.skip_connection and i > 0:
                x = x + x_prev
            x_prev = x

        x = self.convs[-1](x, edge_index, edge_weight, batch)
        return self.unwind(torch.cat([x.real, x.imag], dim=-1))

#### Numerical Visualizations with SymPy

We next wish to test out `MagNet` and verify that it produces coherent outputs. We will instantiate a raw MagNet and run it on all graphs.

In [11]:
# Prepare a graph from the data to be used in the GNN.
load_ex_graph = False

if load_ex_graph:
    with open(save_path, 'rb') as file:
        ex_graph = pickle.load(file)[4]
        N = ex_graph.num_nodes
else:
    ex_graph = utils.scene_graph_dict_to_pyg(eval_data[graph_file][0][2], 'binary')

    N, D = ex_graph.num_nodes, 1024
    ex_graph.edge_index = ex_graph.edge_index.to(device)

    EPS = 1e-12
    MAX_LENGTH = 128
    g = to_networkx(ex_graph, to_undirected=True)
    ex_graph.nxg = g
    all_pairs = dict(nx.all_pairs_dijkstra(g, weight=None))
    delta_max = max(len(path) for target in all_pairs.values() for path in target[1].values())
    paths = torch.full((N, N, delta_max if delta_max < MAX_LENGTH else MAX_LENGTH), -1).long()
    dist = torch.full((N, N), float('inf'))
    for u, (lengths_u, paths_u) in all_pairs.items():
        for v, p in paths_u.items():
            dist[u, v] = lengths_u[v]
            p = (
                torch.tensor(p, device=device).long() if len(p) < MAX_LENGTH 
                else torch.full((MAX_LENGTH,), -1, device=device).long()
            )
            paths[u, v, 0:len(p)] = p
            paths[v, u, 0:len(p)] = p.flip(0)
    dist.fill_diagonal_(EPS)
    ex_graph.diameter = delta_max
    ex_graph.paths = paths.to(device)
    ex_graph.dist = dist.to(device)

    # Topology only: BFS hop counts, never weighted, so metric distances in `dist`
    # can never reach the blurry-vision mask.
    hops = torch.full((N, N), float('inf'))
    for u, lengths_u in nx.all_pairs_shortest_path_length(g):
        for v, h in lengths_u.items():
            hops[u, v] = h
    assert torch.equal(hops[hops.isfinite()], hops[hops.isfinite()].round()), \
        'graph.hops must stay integral (unweighted BFS)'
    ex_graph.hops = hops.to(device)

    # Dense adjacency (bool).
    ex_graph.adj = to_dense_adj(
        ex_graph.edge_index, max_num_nodes=N
    ).squeeze(0).bool().to(device)

# Show the shortest paths matrix of a node in the graph.
node1 = random.randint(0, N - 1)
node2 = random.randint(0, N - 1)
render_matrix(ex_graph.paths[node1, node2][None, :], sig_figs=0)

Matrix([[5, 23, 22, -1, -1, -1]])

In [12]:
# Instantiate a GNN. `model_type` / `model_hparams` are exposed at module scope so
# init_wandb can log the GNN config; create_gnn writes model_hparams as it builds.
def create_gnn(model_type: str):
    global model_hparams
    if model_type == 'gt':
        model_hparams = dict(
            num_layers=3,
            pe_hidden_channels=256,
            pe_num_layers=5,
            d_model=1024,
            heads=8,
            num_samples=320,
            dropout=0.1,
            k_pe=3,
            k_gt=2,
            eps=1e-6,
            use_layer_norm=True,
            directed=True
        )
        gnn = GraphTransformer(**model_hparams)
        gnn.out_features = gnn.d_model
    else:
        model_hparams = dict(
            pe_hidden_channels=256,
            pe_num_layers=5,
            d_model=1024,
            num_samples=320,
            dropout=0.1,
            k=3,
            eps=1e-6,
            use_layer_norm=True,
            directed=True
        )
        gnn = RandomGNNPositionalEncodings(**model_hparams)
        gnn.out_features = gnn.output_projection.out_features
    return gnn


model_type = 'gt'
gnn = create_gnn(model_type)
gnn

GraphTransformer(
  (pe_model): RandomGNNPositionalEncodings(
    (pe_gcn): MagNet(
      (convs): ModuleList(
        (0): MagChebConv(1, 256, K=3, normalization=sym)
        (1-4): 4 x MagChebConv(256, 256, K=3, normalization=sym)
      )
      (norms): ModuleList(
        (0-2): 3 x LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      )
      (unwind): Linear(in_features=512, out_features=256, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (output_projection): Linear(in_features=256, out_features=1024, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  )
  (blocks): ModuleList(
    (0-1): 2 x SparseTransformerBlock(
      (attn): SparseGraphAttention(
        (W_Q): Linear(in_features=1024, out_features=1024, bias=False)
        (W_K): Linear(in_features=1024, out_features=1024, bias=False)
        (W_V): Linear(in_features=1024, out_features=1024, bias=False)
        (dropout): Dro